In [8]:
import requests
import json
import time
import re
from bs4 import BeautifulSoup
from datetime import datetime

BASE_URL = "https://api.cloudflare.riftbound.uvsgames.com/hydraproxy/api/v2"

PLAYERS_FILE = "players.json"

def fetch_event_metadata(event_id):
    url = f"https://locator.riftbound.uvsgames.com/events/{event_id}"
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    # Event name
    name_el = soup.find("h1", {"data-testid": "event-title"})
    name = name_el.text.strip() if name_el else None

    # Date — span.font-medium inside a div that contains a lucide-calendar svg
    date = None
    for svg in soup.find_all("svg", class_=lambda c: c and "lucide-calendar" in c):
        span = svg.find_next_sibling("span", class_="font-medium")
        if span:
            raw = span.text.strip()  # e.g. "Nov 10, 2025"
            date = datetime.strptime(raw, "%b %d, %Y").strftime("%Y-%m-%d")
            break

    # Location — span.font-medium inside the store page link
    location = None
    store_link = soup.find("a", {"aria-label": lambda v: v and v.startswith("View store page")})
    if store_link:
        span = store_link.find("span", class_="font-medium")
        location = span.text.strip() if span else None
        
    tier = "locals"
    for div in soup.find_all("div", class_="text-sm text-white font-semibold uppercase mb-1"):
        if div.text.strip() == "Tournament Format":
            format_div = div.find_next_sibling("div")
            if format_div and "Modified Champion Deck" in format_div.text:
                tier = "release"
            break

    n_rounds = None
    for trigger in soup.find_all("button", {"data-testid": "pairings-round-dropdown-trigger"}):
        span = trigger.find("span", {"data-slot": "select-value"})
        if span:
            match = re.search(r"Round\s+(\d+)", span.text.strip())
            if match:
                n_rounds = int(match.group(1))
                break
    
    return {
        "event_id": str(event_id),
        "name": name,
        "date": date,
        "location": location,
        "tier": tier,
        "n_rounds": n_rounds,
    }

def register_event(event_id, final_round_id):
    metadata = fetch_event_metadata(event_id)
    metadata["final_round_id"] = final_round_id
    metadata["scraped"] = False

    with open("events.json") as f:
        events = json.load(f)

    events.append(metadata)

    with open("events.json", "w") as f:
        json.dump(events, f, indent=2)

    print(f"Registered: {metadata['name']} ({metadata['date']}) @ {metadata['location']} [{metadata['tier']}] — {metadata['n_rounds']} rounds ending at round {final_round_id}")


# ── USAGE ────────────────────────────────────────────────────────────────────

events = []

# Add each event: name, location, date, tier, first_round_id
# Get first_round_id from DevTools: it's the lowest round ID you see for that event
with open("events.json") as f:
    events = json.load(f)

all_pairings = []
for event in events:
    if event.get("scraped"):
        continue
    data = scrape_event(
        event_id=event["event_id"],
        event_name=event["name"],
        location=event["location"],
        date=event["date"],
        tier=event.get("tier", "locals"),  # default to "locals" if missing
    )
    all_pairings.append(data)
    event["scraped"] = True

# Write back updated scraped flags
with open("events.json", "w") as f:
    json.dump(events, f, indent=2)

with open("riftbound_pairings.json", "w") as f:
    json.dump(all_pairings, f, indent=2)

print("Saved to riftbound_pairings.json")

Scraping: Nexus Night - Halloween Release! 👻🌌


NotImplementedError: 

In [9]:
# Reg tab
register_event(237336,141705)
register_event(193622,144228)
register_event(199204,144190)
register_event(207462,146242)
register_event(193659,148685)
register_event(202320,155990)
register_event(246742,156182)
register_event(247882,165241)
register_event(207469,168426)
register_event(224858,175468)

None
Registered: Nexus Night - Halloween Release! 👻🌌 (2025-10-31) @ Otter space [release] — None rounds ending at round 141705
None
Registered: Saturday Afternoon Release Event (2025-11-01) @ Underworld Gaming [release] — None rounds ending at round 144228
None
Registered: Rift bound Release event for Gamers World! (2025-11-01) @ Gamers World, Ireland [release] — None rounds ending at round 144190
None
Registered: Saturday Morning Release Event - Late Registration (2025-11-01) @ ReRoll Games [release] — None rounds ending at round 146242
None
Registered: Sunday Afternoon Release Event (2025-11-02) @ Underworld Gaming [release] — None rounds ending at round 148685
None
Registered: Release Event - Late Registration (2025-11-06) @ Dungeons and Donuts [release] — None rounds ending at round 155990
None
Registered: Otter Nexus Night (2025-11-06) @ Otter space [locals] — None rounds ending at round 156182
None
Registered: Saturday Afternoon Nexus Night (2025-11-08) @ Gamers World, Ireland [l

In [31]:
!pip install nest_asyncio

In [3]:
import requests
import re

def get_round_info_from_html(event_id, n_rounds):
    url = f"https://locator.riftbound.uvsgames.com/events/224858"
    r = requests.get(url)
    
    # Look for any number that appears near "tournament_round" or "round" in script tags
    # Also try to find round IDs embedded in JSON state blobs
    matches = re.findall(r'"tournament.?round[s]?"[:\s]+(\d+)', r.text, re.IGNORECASE)
    if not matches:
        matches = re.findall(r'"round.?id"[:\s]+(\d+)', r.text, re.IGNORECASE)
    
    print(f"Candidate round IDs found in HTML: {matches}")
    return matches

In [ ]:
async def get_round_info_async(event_id):
    final_round_id = None

    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()

        def handle_request(request):
            nonlocal final_round_id
            match = re.search(r"/tournament-rounds/(\d+)/matches/paginated/", request.url)
            if match:
                final_round_id = int(match.group(1))

        page.on("request", handle_request)
        await page.goto(f"https://locator.riftbound.uvsgames.com/events/{event_id}")
        await page.wait_for_load_state("networkidle")
        html = await page.content()
        await browser.close()

    soup = BeautifulSoup(html, "html.parser")
    n_rounds = None
    for trigger in soup.find_all("button", {"data-testid": "pairings-round-dropdown-trigger"}):
        span = trigger.find("span", {"data-slot": "select-value"})
        if span:
            match = re.search(r"Round\s+(\d+)", span.text.strip())
            if match:
                n_rounds = int(match.group(1))
                break

    if final_round_id is None or n_rounds is None:
        raise ValueError(f"Could not determine round info for event {event_id}")

    first_round_id = final_round_id - (n_rounds - 1)
    return first_round_id, n_rounds

def get_round_info(event_id):
    return asyncio.get_event_loop().run_until_complete(get_round_info_async(event_id))


def scrape_event(event_id, event_name, date, tier, location):
    print(f"Scraping: {event_name}")

    first_round_id, n_rounds = get_round_info(event_id)
    print(f"  {n_rounds} rounds, IDs {first_round_id}–{first_round_id + n_rounds - 1}")

    registry = load_registry()
    rounds = []

    for i in range(n_rounds):
        round_id = first_round_id + i
        round_num = i + 1
        print(f"  Round {round_num} (id={round_id})...", end=" ")

        raw_matches = fetch_round(round_id)
        pairings = [parse_match(m) for m in raw_matches]
        pairings = [p for p in pairings if p is not None]

        for raw_match in raw_matches:
            update_registry(registry, raw_match)

        print(f"{len(pairings)} pairings")
        rounds.append({"round": round_num, "pairings": pairings})
        time.sleep(0.3)

    save_registry(registry)

    return {
        "event_id": str(event_id),
        "name": event_name,
        "date": date,
        "tier": tier,
        "location": location,
        "n_rounds": n_rounds,
        "rounds": rounds
    }